# FDMNES Pipeline Demo (K-edge/CIF and L-edge/XYZ)

A thin, interactive wrapper around the `fdmnes_pipeline` package, covering
both structure input modes it supports:

1. **K-edge / CIF** -- a crystal structure from a `.cif` file (e.g. the
   Ni-HHTP/Co-HHTP 2D MOF workflow this package was originally built for).
2. **L-edge / XYZ** -- an isolated molecule placed directly in the input
   file via FDMNES's `Molecule` block, parsed from a plain `.xyz` file.

Stages covered:
1. Generate input file(s) from a `JobSpec` + `ProtocolSettings`.
2. (Optional, only once FDMNES is installed on *this* machine) run FDMNES
   and validate the outputs.

If you're running this on a machine where FDMNES isn't installed, stop
after step 1 -- `run_single`/`run_batch` need a real executable path.
The structures used below (`examples/structures/demo_*.cif`/`.xyz`) are
synthetic placeholders (an idealized `Ni-O6` octahedron and idealized
octahedral `Cu(H2O)6`/`Zn(H2O)6` complexes) chosen only so this notebook
runs out of the box -- swap in your own CIF/XYZ for real work.


## 0. Setup

Make sure the package is importable (either `pip install -e .` from the repo root, or run this notebook from the repo root).

In [ ]:
import sys
from pathlib import Path

# If running the notebook straight from notebooks/, add the repo root to sys.path.
repo_root = Path.cwd().parent if (Path.cwd() / "fdmnes_pipeline").exists() is False else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from fdmnes_pipeline import (
    ProtocolSettings, JobSpec, generate_input_file, generate_batch,
    FdmnesRunner, validate_run, validate_batch,
)
from fdmnes_pipeline.validation import report_table


## 1. Define the standardized protocol

`ProtocolSettings` holds everything shared across jobs: radius, energy
range, and the Green/FDM, SCF, PBE96, Quadrupole and Convolution toggles.
These values match the ones validated on this project's own Co-HHTP
K-edge run (`radius=5.0`, this energy grid, Convolution on).

In [ ]:
protocol = ProtocolSettings(
    radius=5.0,
    energy_range="-5.0 0.2 10.0 0.5 25.0 1.0 60.0 2.0 100.0",
    use_green=False,
    use_pbe96=False,
    use_scf=False,
    use_quadrupole=False,
    use_convolution=True,
)
protocol


## 2. Define one job and generate its input file

This uses the synthetic demo CIF bundled with the repo, so it runs as-is.
For real use, point `structure_file`/`filout`/`input_file` at your own CIF
and FDMNES working directory.

In [ ]:
job = JobSpec(
    name="Demo_Ni_MO6",
    structure_file=repo_root / "examples" / "structures" / "demo_MO6.cif",
    element="Ni",
    absorber_mode="all",
    absorber_index=1,
    filout=repo_root / "examples" / "outputs" / "Demo_Ni_MO6" / "Demo_Ni_MO6_FDM",
    input_file=repo_root / "examples" / "outputs" / "Demo_Ni_MO6" / "Demo_Ni_MO6_input.txt",
)

text = generate_input_file(job, protocol)
print(text)


## 3. Generate a batch (several structures at once)

You can also load a batch straight from a JSON config with
`fdmnes_pipeline.batch_config.load_config` -- see `examples/jobs.json`,
which has this same demo job plus a `"Your_Structure_Here"` template to
duplicate for your own structures.

In [ ]:
from fdmnes_pipeline.batch_config import load_config

protocol2, jobs, per_job_overrides = load_config(repo_root / "examples" / "jobs.json")
generate_batch(jobs, protocol2, per_job_protocol=per_job_overrides)


## 4. Run FDMNES (only once it's installed on this machine)

FDMNES itself is **not** bundled with this package -- download it from
Institut Néel/CNRS (see the main README) and point `executable` at it.
Start with a generous timeout: XANES calculations with SCF and a large
radius can run for a long time, and a too-tight timeout looks identical
to a real hang in the validation report.

In [ ]:
# runner = FdmnesRunner(
#     executable=Path("/path/to/fdmnes_linux64"),
#     timeout_seconds=7200,
# )
# results = runner.run_batch(jobs)
# for job_, result in zip(jobs, results):
#     print(job_.name, "launched:", result.launched, "returncode:", result.returncode,
#           "timed_out:", result.timed_out, f"({result.elapsed_seconds:.1f}s)" if result.elapsed_seconds else "")


## 5. Validate outputs

Produces a `job_name -> SUCCESS/FAILED/WARNING` table; see the main
README's "Validation logic" section for exactly what each status checks.

In [ ]:
# validation_results = validate_batch(jobs, run_results=results)
# print(report_table(validation_results))

# Or validate files from a run that already happened outside this notebook:
validation_results = validate_batch(jobs)
print(report_table(validation_results))


## 6. L-edge jobs from a .xyz structure

A second workflow: instead of a CIF + K-edge, these jobs use an isolated
molecule parsed straight from a `.xyz` file (`xyz_parser.parse_xyz` --
no `ase` dependency) and select the L23 edge. Every field below (`edge`,
`molecule`, `use_green`, `vmax`, `use_tddft`, `use_spinorbit`) is additive:
a plain K-edge/CIF job like the ones above is completely unaffected by any
of it. `demo_Cu_hexaaqua.xyz` is the same synthetic octahedral Cu(H2O)6
molecule used in the test suite's input-format regression test.

In [ ]:
from fdmnes_pipeline import parse_xyz

molecule = parse_xyz(repo_root / "examples" / "structures" / "demo_Cu_hexaaqua.xyz")
print(f"{len(molecule.atoms)} atoms, cell = {molecule.cell}")

demo_ledge_protocol = ProtocolSettings(
    radius=7.5,
    energy_range="-10. 0.1 15. 0.2 35. 0.5 60.",
    use_green=True,
    vmax=-6.0,
    use_tddft=True,
    use_spinorbit=True,
)

demo_ledge_job = JobSpec(
    name="Demo_Cu_hexaaqua",
    element="Cu",
    edge="L23",
    molecule=molecule,
    filout=repo_root / "examples" / "outputs" / "Demo_Cu_hexaaqua" / "Demo_Cu_hexaaqua_FDM",
    input_file=repo_root / "examples" / "outputs" / "Demo_Cu_hexaaqua" / "Demo_Cu_hexaaqua_input.txt",
)

text = generate_input_file(demo_ledge_job, demo_ledge_protocol, warn_if_structure_missing=False)
print(text[:600])


Or load an L-edge batch straight from JSON, same as the K-edge example --
see `examples/jobs_ledge.json`, which points at the synthetic demo `.xyz`
files under `examples/structures/`, so this runs as-is:

In [ ]:
ledge_protocol, ledge_jobs, ledge_overrides = load_config(repo_root / "examples" / "jobs_ledge.json")
generate_batch(ledge_jobs, ledge_protocol, per_job_protocol=ledge_overrides)
